# Treinando um agente para jogar **Breakout** (Atari) com PPO

**Disciplina:** Aprendizado por Reforço — IBMEC 2026/2 · **Professor:** Pedro Calais
**Atividade Complementar Final** · **Aluno(s):** _preencher_

Este notebook é a entrega (item **m**). O código fica no pacote `src/breakout_rl/`, que este
notebook importa. O treino longo (~4 h num Mac M3) roda **fora** do notebook pelo
`scripts/train_overnight.sh`, com checkpoints e retomada. Aqui carregamos os resultados.

| Item | Seção |
|---|---|
| a, b | 1. Jogo escolhido |
| c, d, e | 2. Ambiente, ações e recompensas |
| f | 3. Política aleatória |
| h | 4. Ajustes em relação ao código base |
| g | 5. Treinamento e TensorBoard |
| i | 6. Tempo até uma política minimamente inteligente |
| j | 7. Parâmetros da rede |
| k, l | 8. Agente treinado e análise das jogadas |

## 0. Setup
Funciona localmente (`make setup`) e no Google Colab (clona o repositório).

In [ ]:
import os, sys, json
from pathlib import Path

if "google.colab" in sys.modules:  # Colab: clona e instala
    if not Path("rl-breakout-ppo").exists():
        !git clone -q https://github.com/vnevesl/rl-breakout-ppo.git
    os.chdir("rl-breakout-ppo")
    !pip install -q -e .
else:  # local: o notebook fica em notebooks/
    if Path.cwd().name == "notebooks":
        os.chdir("..")
sys.path.insert(0, str(Path("src").resolve()))
ROOT = Path.cwd()
RUN = ROOT / "runs" / "ppo_breakout"
print("Raiz do projeto:", ROOT)

In [ ]:
from IPython.display import Video, Image, Markdown, display
import pandas as pd

from breakout_rl.config import ExperimentConfig
from breakout_rl.envs import describe_env

cfg = ExperimentConfig.from_yaml("configs/ppo_breakout.yaml")

## 1. Jogo escolhido (a, b)

**a) Breakout** (`ALE/Breakout-v5`, Atari 2600, 1978). O jogador controla uma raquete na base da
tela e rebate uma bola contra um muro de tijolos coloridos. Cada tijolo destruído dá pontos.
O objetivo é destruir o muro sem deixar a bola passar pela raquete.

**b) Jogando você mesmo:** `make play` (ou `python -m breakout_rl.play`). Use ← → para mover e
ESPAÇO (FIRE) para lançar a bola.

> _Impressões ao jogar (preencher):_ ex.: a bola acelera depois de alguns toques e ao atingir
> as fileiras de cima; a raquete encolhe quando a bola chega ao teto; é preciso apertar FIRE a
> cada vida perdida.

## 2. Ambiente, ações e recompensas (c, d, e)

In [ ]:
info = describe_env("ALE/Breakout-v5")
info

**c) Ambiente.** Vem do Arcade Learning Environment (ALE) via Gymnasium.
- **Observação crua:** imagem RGB `210×160×3` (`Box(0, 255, (210,160,3), uint8)`). Também existem
  as versões `grayscale` e `ram` (128 bytes da memória do Atari).
- **Dinâmica no v5:** `frameskip=4` (cada ação dura 4 frames) e `repeat_action_probability=0.25`
  ("sticky actions": 25% de chance de repetir a ação anterior, o que dá estocasticidade).
- **Episódio:** 5 vidas. Termina quando as vidas acabam (ou no limite de 108.000 frames).
- **Pré-processamento usado no treino** (`AtariWrapper` do SB3 + `VecFrameStack`):
  até 30 no-ops aleatórios no início; ação repetida por 4 frames (max dos 2 últimos);
  FIRE automático no reset; escala de cinza 84×84 ("se livre de pixels inúteis");
  perda de vida tratada como fim de episódio no treino; recompensa clipada em {-1, 0, +1};
  **4 frames empilhados** para que a rede perceba a direção e a velocidade da bola.
  A observação final tem formato `84×84×4`.

**d) Ações: 4 ações discretas** (`Discrete(4)`, conjunto mínimo):

| id | ação | efeito |
|---|---|---|
| 0 | NOOP | não faz nada |
| 1 | FIRE | lança a bola (necessário no início e após cada vida perdida) |
| 2 | RIGHT | move a raquete para a direita |
| 3 | LEFT | move a raquete para a esquerda |

Com `full_action_space=True` o ALE expõe as 18 ações do controle do Atari, mas as 14 extras são
redundantes neste jogo.

**e) Recompensas.** O agente ganha pontos ao destruir tijolos, e o valor depende da cor/fileira
(manual do Atari): as **2 fileiras de baixo (azul, verde) valem 1**, as **2 do meio (amarela,
laranja) valem 4** e as **2 de cima (laranja-escuro, vermelha) valem 7**. Um muro completo vale
432 pontos e, ao limpá-lo, surge um segundo muro (máximo teórico de 864). Não há recompensa
negativa: perder a bola só custa uma vida, o que encerra a partida mais cedo e reduz a pontuação
futura.
No treino, a recompensa é **clipada** para +1 por tijolo, para estabilizar o aprendizado. Os
gráficos e as avaliações mostram a **pontuação real** do jogo.

## 3. Política aleatória, sem aprendizado (f)

In [ ]:
from breakout_rl.record import record
from breakout_rl.evaluate import evaluate

random_video = Path("reports/videos/random_policy.mp4")
if not random_video.exists():
    record("random", random_video, cfg.env, episodes=3)
display(Video(str(random_video), embed=True, width=320))

In [ ]:
rand = evaluate("random", cfg.env, episodes=20)
print(f"Política aleatória: {rand['mean']:.1f} ± {rand['std']:.1f} pontos por partida "
      f"(min {rand['min']:.0f}, max {rand['max']:.0f}) em 20 partidas")

A política aleatória quase nunca acerta a bola: perde as 5 vidas em poucos segundos e faz ~1–2
pontos por partida. Esse é o *baseline* que o agente precisa superar.

## 4. Ajustes em relação ao código base (h)

O código base (Freeway, "jogo da galinha") treinava PPO com 1 ambiente, sem empilhar frames, por
50 mil passos. O Freeway é fácil: basta "andar para cima" com timing. O Breakout exige
**rastrear a bola** e **planejar a posição da raquete**, então ajustamos o seguinte (valores
inspirados nos hiperparâmetros de Atari do RL Baselines3 Zoo):

| Parâmetro | Código base | Este trabalho | Por quê |
|---|---|---|---|
| `total_timesteps` | 50.000 | **10.000.000** | Breakout precisa de muito mais experiência |
| ambientes paralelos (`n_envs`) | 1 | **8** | mais experiência variada por atualização e por segundo |
| frame stack | nenhum | **4** | 1 frame não mostra a direção/velocidade da bola |
| `frameskip` do `gym.make` | 4 (padrão v5) | **1** | o `AtariWrapper` já pula 4; com os dois juntos, o agente via 1 a cada 16 frames |
| `repeat_action_probability` | 0.25 | **0.0** | igual ao `BreakoutNoFrameskip-v4` dos benchmarks |
| `n_steps` (por env) | 2048 | **128** | 128×8 = 1024 passos por atualização, atualizações mais frequentes |
| `batch_size` | 256 | 256 | igual |
| `n_epochs` | 10 | **4** | menos reuso do mesmo lote evita políticas instáveis |
| `learning_rate` | 2.5e-4 constante | **2.5e-4 → 0 (linear)** | passos grandes no início, ajuste fino no fim |
| `clip_range` | 0.2 | **0.1 → 0 (linear)** | atualizações de política mais conservadoras |
| `ent_coef` | 0.0 | **0.01** | bônus de entropia mantém a exploração |
| `device` | cpu | **mps** (GPU Apple) | ~760 contra ~255 passos/s no M3, cerca de 3× mais rápido |

Esses valores estão versionados em `configs/ppo_breakout.yaml`. A configuração do professor
aplicada ao Breakout fica em `configs/baseline_professor.yaml`, para comparação.

In [ ]:
import yaml
print(yaml.safe_dump(cfg.to_dict(), sort_keys=False, allow_unicode=True))

## 5. Treinamento e TensorBoard (g)

O treino roda em segundo plano, fora do notebook:

```bash
./scripts/train_overnight.sh        # caffeinate + nohup + --resume; checkpoint a cada 250k passos
tail -f runs/ppo_breakout/train.log
./scripts/tensorboard.sh            # http://localhost:6006
./scripts/stop_training.sh          # para salvando checkpoint; rode o 1º comando para retomar
```

Para um teste rápido dentro do notebook, descomente a célula abaixo (≈1 min).

In [ ]:
# from breakout_rl.train import train
# smoke = ExperimentConfig.from_yaml("configs/smoke.yaml")
# train(smoke)

**Print do TensorBoard:** depois do treino, abra o TensorBoard, tire o print do painel
`rollout/ep_rew_mean` e salve em `reports/figures/tensorboard.png`.
Métricas principais:
- `rollout/ep_rew_mean`: pontuação média por partida (últimas 100). É a curva de aprendizado.
- `eval/mean_reward`: pontuação na avaliação periódica (a cada 250 mil passos).
- `train/entropy_loss`: cai conforme a política fica mais confiante.
- `train/approx_kl` e `train/clip_fraction`: tamanho das atualizações do PPO.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
tb_png = Path("reports/figures/tensorboard.png")
if tb_png.exists():
    display(Image(str(tb_png)))

from breakout_rl.plots import plot_run
if RUN.exists():
    for p in plot_run(RUN, Path("reports/figures")):
        display(Image(str(p)))
else:
    print("Treino ainda não executado.")

## 6. Quanto tempo até uma política minimamente inteligente? (i)

In [ ]:
ms_path = RUN / "milestones.json"
if ms_path.exists():
    ms = json.loads(ms_path.read_text())
    df = pd.DataFrame(ms["milestones"]).T.rename_axis("média ≥").reset_index()
    display(df)
    print(f"Tempo total de treino: {ms['elapsed_seconds']/3600:.2f} h, "
          f"{ms.get('last_timesteps', 0):,} passos")
else:
    print("Rode o treino primeiro.")

**Critério de "minimamente inteligente":** média ≥ **30 pontos** por partida nas últimas 100
partidas, ou seja, ~15× a política aleatória. Nesse nível o agente já segue a bola com a
raquete de forma consistente.

> _Resposta (preencher com a tabela acima):_ o agente atingiu média 30 com **X milhões de passos
> (~Y h)** num MacBook M3 (GPU via MPS).

## 7. Quantos parâmetros tem a rede? (j)

In [ ]:
from stable_baselines3 import PPO
from breakout_rl.model_info import summarize

model_path = RUN / "best_model" / "best_model.zip"
if model_path.exists():
    policy = PPO.load(model_path, device="cpu").policy
else:
    from breakout_rl.model_info import _fresh_policy
    policy = _fresh_policy()
s = summarize(policy)
display(pd.DataFrame(s["layers"], columns=["camada", "shape", "parâmetros"]))
print(f"TOTAL: {s['total']:,} parâmetros")

A `CnnPolicy` do SB3 é a "Nature CNN" (Mnih et al., 2015), compartilhada entre ator e crítico:

| Bloco | Camada | Parâmetros |
|---|---|---|
| Extrator | Conv 8×8, 32 filtros, stride 4 (entrada 4 canais) | 8.224 |
| | Conv 4×4, 64 filtros, stride 2 | 32.832 |
| | Conv 3×3, 64 filtros, stride 1 | 36.928 |
| | Linear 3136 → 512 | 1.606.144 |
| Ator (`action_net`) | Linear 512 → 4 (uma saída por ação) | 2.052 |
| Crítico (`value_net`) | Linear 512 → 1 (V(s)) | 513 |
| **Total** | | **1.686.693** |

95% dos parâmetros estão na camada densa que liga as convoluções às 512 features.

## 8. Agente após o treinamento (k, l)

In [ ]:
trained_video = Path("reports/videos/trained_policy.mp4")
if model_path.exists():
    if not trained_video.exists():
        record(str(model_path), trained_video, cfg.env, episodes=2, max_steps=10_000)
    display(Video(str(trained_video), embed=True, width=320))
    trained = evaluate(str(model_path), cfg.env, episodes=20)
    print(f"Agente treinado: {trained['mean']:.1f} ± {trained['std']:.1f} pontos "
          f"(max {trained['max']:.0f}) vs aleatório {rand['mean']:.1f}")
else:
    print("Rode o treino primeiro.")

In [ ]:
# Linha do tempo do vídeo: cada tijolo e cada vida perdida, com o segundo exato no vídeo.
ev_path = Path("reports/videos/trained_policy_events.csv")
if ev_path.exists():
    ev = pd.read_csv(ev_path)
    display(ev[ev.event != "tijolo"])            # vidas perdidas / travamentos
    display(ev[ev.event == "tijolo"].head(30))   # primeiros tijolos

**l) Jogada "inteligente" e jogada "burra"** (preencher assistindo ao vídeo; o CSV acima dá os
instantes):

| Tipo | Instante no vídeo | O que aconteceu |
|---|---|---|
| Inteligente | `t = __ s` | _ex.: o agente abre um túnel na lateral do muro e manda a bola para cima, onde ela quebra vários tijolos sozinha (estratégia clássica descoberta pelo DQN)._ |
| Burra | `t = __ s` | _ex.: depois que a bola acelera, a raquete chega atrasada e o agente perde uma vida numa bola "fácil"; ou fica parado sem apertar FIRE._ |

## 9. Conclusões
- _preencher: curva de aprendizado, comparação com o aleatório, limitações (ex.: dificuldade
  com a bola rápida), o que faria com mais tempo (mais passos, DQN para comparar, sticky actions)._

### Referências
- Stable-Baselines3: https://stable-baselines3.readthedocs.io
- RL Baselines3 Zoo, hiperparâmetros de Atari: https://github.com/DLR-RM/rl-baselines3-zoo
- ALE Breakout: https://ale.farama.org/environments/breakout/
- Schulman et al. (2017), *Proximal Policy Optimization Algorithms*
- Mnih et al. (2015), *Human-level control through deep reinforcement learning*